# Chapter 6: Multiple Regression Analysis — Further Issues

Topics: scaling, beta coefficients, logarithms, polynomials, prediction intervals, interaction terms.

## Step 1 – Setup: Working Directory & Output Folder

Run this cell first to fix the working directory and create the `output/` folder.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)
df = woo.dataWoo('wage1')

## 6.1.3 Logarithms

## Step 2 – Scaling and Standardised (Beta) Coefficients

Changing units of $x_j$ by a factor $c$ multiplies $\hat{\beta}_j$ by $1/c$ but leaves t-statistics and R² unchanged.

**Standardised beta coefficient** (allows comparison across variables):
$$\hat{b}_j = \hat{\beta}_j \cdot \frac{s_{x_j}}{s_y}$$

Interpretation: a one-standard-deviation increase in $x_j$ changes $y$ by $\hat{b}_j$ standard deviations.

In [ ]:
# Log-level model
m_log = smf.ols('np.log(wage) ~ educ + exper + tenure', data=df).fit()
print(m_log.summary())
print(f"\nReturns to education: {m_log.params['educ']*100:.2f}% per year")

## 6.1.4 Quadratics and Polynomials

## Step 3 – Logarithmic Functional Forms

| Model | Equation | $\partial y / \partial x$ interpretation |
|-------|----------|--------------------------------------------|
| Level-level | $y=\beta_0+\beta_1 x+u$ | $\Delta y \approx \beta_1 \Delta x$ |
| Log-level | $\log y=\beta_0+\beta_1 x+u$ | $\%\Delta y \approx 100\beta_1 \Delta x$ |
| Level-log | $y=\beta_0+\beta_1\log x+u$ | $\Delta y \approx (\beta_1/100)\%\Delta x$ |
| Log-log | $\log y=\beta_0+\beta_1\log x+u$ | $\%\Delta y \approx \beta_1 \%\Delta x$ (elasticity) |

Use the **Adjusted R²** to compare models with the same dependent variable (do **not** compare R² across log vs. level models directly):
$$\bar{R}^2 = 1 - \frac{SSR/(n-k-1)}{SST/(n-1)}$$

In [ ]:
m_quad = smf.ols('np.log(wage) ~ educ + exper + I(exper**2) + tenure', data=df).fit()
print(m_quad.summary())

# Turning point: d(log wage)/d(exper) = b_exper + 2*b_exper2*exper = 0
b1 = m_quad.params['exper']
b2 = m_quad.params['I(exper ** 2)']
turning_point = -b1 / (2 * b2)
print(f"\nTurning point of experience: {turning_point:.1f} years")

# Plot marginal effect
exper_range = np.linspace(0, 40, 100)
marginal = b1 + 2 * b2 * exper_range
plt.figure(figsize=(8, 4))
plt.plot(exper_range, marginal, color='steelblue', lw=2)
plt.axhline(0, color='red', linestyle='--')
plt.axvline(turning_point, color='green', linestyle='--', label=f'Turning point: {turning_point:.1f} yrs')
plt.xlabel('Experience (years)'); plt.ylabel('Marginal effect on log(wage)')
plt.title('Marginal Effect of Experience (Quadratic Model)')
plt.legend()
plt.savefig(f"{output_dir}/fig1_quadratic_exper.png", dpi=150)
plt.savefig(os.path.join(output_dir, 'fig_Chapter_06_Multiple_Regression_Further_Issues_01.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6.1.6 Interaction Terms

## Step 4 – Quadratic Models and Turning Points

A **quadratic model** allows for a non-linear relationship:
$$y = \beta_0 + \beta_1 x + \beta_2 x^2 + u$$

The **marginal effect** of $x$ on $y$:
$$\frac{\partial y}{\partial x} = \beta_1 + 2\beta_2 x$$

**Turning point** (maximum or minimum):
$$x^* = -\frac{\hat{\beta}_1}{2\hat{\beta}_2}$$

If $\hat{\beta}_2 < 0$: inverted U-shape (e.g., wages peaking at some experience level).
If $\hat{\beta}_2 > 0$: U-shape.

In [ ]:
m_interact = smf.ols('np.log(wage) ~ educ * female + exper + tenure', data=df).fit()
print(m_interact.summary())

b_educ        = m_interact.params['educ']
b_female_educ = m_interact.params['educ:female']
print(f"\nReturn to education for men:   {b_educ*100:.2f}%")
print(f"Return to education for women: {(b_educ + b_female_educ)*100:.2f}%")

## 6.2 Prediction and Confidence Intervals

## Step 5 – Interaction Terms

An **interaction term** allows the effect of one variable to depend on another:
$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \beta_3 (x_1 \cdot x_2) + u$$

**Partial effect** of $x_1$:
$$\frac{\partial y}{\partial x_1} = \beta_1 + \beta_3 x_2$$

This effect **varies with the level of $x_2$** — e.g., the return to education may differ by gender if $x_1 = educ$ and $x_2 = female$.

In [ ]:
new_obs = pd.DataFrame({'educ': [16], 'exper': [5], 'tenure': [2]})
pred = m_log.get_prediction(new_obs)
pred_df = pred.summary_frame(alpha=0.05)
print("Prediction for: educ=16, exper=5, tenure=2")
print(pred_df[['mean','mean_ci_lower','mean_ci_upper','obs_ci_lower','obs_ci_upper']])
print(f"\nPredicted log(wage): {pred_df['mean'].values[0]:.4f}")
print(f"Predicted wage:      ${np.exp(pred_df['mean'].values[0]):.2f}/hr")

## Step 6 – Prediction and Confidence Intervals

For a new observation $\mathbf{x}_0$:

**Point prediction:** $\hat{y}_0 = \mathbf{x}_0'\hat{\boldsymbol{\beta}}$

**95% confidence interval for $E(y|\mathbf{x}_0)$** (mean response):
$$\hat{y}_0 \pm t_{0.025,\,n-k-1} \cdot \hat{\sigma}\sqrt{\mathbf{x}_0'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{x}_0}$$

**95% prediction interval for a single new observation:**
$$\hat{y}_0 \pm t_{0.025,\,n-k-1} \cdot \hat{\sigma}\sqrt{1 + \mathbf{x}_0'(\mathbf{X}'\mathbf{X})^{-1}\mathbf{x}_0}$$

Prediction intervals are always wider than confidence intervals due to the extra uncertainty about the individual error term.